In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

In [2]:
# Input and output file paths
INPUT_FILE = "../data/snapshots.csv"
OUTPUT_FILE = "../data/analysis.csv"

# Date ranges we'll use for analysis
TODAY = datetime.today().strftime("%Y-%m-%d")
SEVEN_DAYS_AGO = (datetime.today() - timedelta(days=7)).strftime("%Y-%m-%d")
FOURTEEN_DAYS_AGO = (datetime.today() - timedelta(days=14)).strftime("%Y-%m-%d")

print(f"Today: {TODAY}")
print(f"7 days ago: {SEVEN_DAYS_AGO}")
print(f"14 days ago: {FOURTEEN_DAYS_AGO}")

Today: 2026-05-01
7 days ago: 2026-04-24
14 days ago: 2026-04-17


In [3]:
# Load snapshots
df = pd.read_csv(INPUT_FILE)

# Convert snapshot_date to proper datetime type
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

# Quick look at what we have
print(f"Total rows: {len(df)}")
print(f"Date range: {df['snapshot_date'].min()} → {df['snapshot_date'].max()}")
print(f"Languages tracked: {df['language'].nunique()}")
df.head()

Total rows: 10
Date range: 2026-05-01 00:00:00 → 2026-05-01 00:00:00
Languages tracked: 10


,language,total_repos,top_repo_name,top_repo_stars,top_repo_url,top_repo_description,snapshot_date
0,Python,110355,NousResearch/hermes-agent,128013,https://github.com/NousResearch/hermes-agent,The agent that grows with you,2026-05-01
1,JavaScript,29834,affaan-m/everything-claude-code,171432,https://github.com/affaan-m/everything-claude-...,The agent harness performance optimization sys...,2026-05-01
2,TypeScript,49258,openclaw/openclaw,367120,https://github.com/openclaw/openclaw,Your own personal AI assistant. Any OS. Any Pl...,2026-05-01
3,Rust,15995,ultraworkers/claw-code,189525,https://github.com/ultraworkers/claw-code,The repo is finally unlocked. enjoy the party!...,2026-05-01
4,Go,13569,danielmiessler/Fabric,41314,https://github.com/danielmiessler/Fabric,Fabric is an open-source framework for augment...,2026-05-01


In [4]:
# Get the most recent snapshot date
latest_date = df["snapshot_date"].max()

# Filter to only today's data
df_latest = df[df["snapshot_date"] == latest_date]

# Rank languages by total repo count
language_ranking = (
    df_latest[["language", "total_repos"]]
    .sort_values("total_repos", ascending=False)
    .reset_index(drop=True)
)

language_ranking.index += 1  # Start ranking from 1
print(f"Language ranking as of {latest_date.strftime('%Y-%m-%d')}:")
language_ranking

Language ranking as of 2026-05-01:


,language,total_repos
1,Python,110355
2,TypeScript,49258
3,JavaScript,29834
4,Rust,15995
5,C++,14954
6,Go,13569
7,Java,8522
8,Kotlin,5258
9,Swift,4623
10,Ruby,1208


In [5]:
# Sort all snapshots by stars to find the hottest repos right now
star_leaderboard = (
    df_latest[["language", "top_repo_name", "top_repo_stars", "top_repo_url", "top_repo_description"]]
    .sort_values("top_repo_stars", ascending=False)
    .reset_index(drop=True)
)

star_leaderboard.index += 1
print("Top repos by star count right now:")
star_leaderboard

Top repos by star count right now:


,language,top_repo_name,top_repo_stars,top_repo_url,top_repo_description
1,TypeScript,openclaw/openclaw,367120,https://github.com/openclaw/openclaw,Your own personal AI assistant. Any OS. Any Pl...
2,Rust,ultraworkers/claw-code,189525,https://github.com/ultraworkers/claw-code,The repo is finally unlocked. enjoy the party!...
3,JavaScript,affaan-m/everything-claude-code,171432,https://github.com/affaan-m/everything-claude-...,The agent harness performance optimization sys...
4,Python,NousResearch/hermes-agent,128013,https://github.com/NousResearch/hermes-agent,The agent that grows with you
5,C++,LadybirdBrowser/ladybird,62618,https://github.com/LadybirdBrowser/ladybird,Truly independent web browser
6,Go,danielmiessler/Fabric,41314,https://github.com/danielmiessler/Fabric,Fabric is an open-source framework for augment...
7,Kotlin,kavishdevar/librepods,26774,https://github.com/kavishdevar/librepods,AirPods liberated from Apple's ecosystem.
8,Swift,apple/container,26263,https://github.com/apple/container,A tool for creating and running Linux containe...
9,Java,opendataloader-project/opendataloader-pdf,19844,https://github.com/opendataloader-project/open...,PDF Parser for AI-ready data. Automate PDF acc...
10,Ruby,antiwork/gumroad,8995,https://github.com/antiwork/gumroad,Sell stuff and see what sticks


In [6]:
# Calculate the share of total repos each language represents
total_repos_all = df_latest["total_repos"].sum()

topic_pulse = df_latest[["language", "total_repos"]].copy()
topic_pulse["share_pct"] = (
    (topic_pulse["total_repos"] / total_repos_all) * 100
).round(2)
topic_pulse = topic_pulse.sort_values("share_pct", ascending=False).reset_index(drop=True)
topic_pulse.index += 1

print(f"Total repos across all tracked languages: {total_repos_all:,}")
print("\nLanguage share of ecosystem:")
topic_pulse

Total repos across all tracked languages: 253,576

Language share of ecosystem:


,language,total_repos,share_pct
1,Python,110355,43.52
2,TypeScript,49258,19.43
3,JavaScript,29834,11.77
4,Rust,15995,6.31
5,C++,14954,5.90
6,Go,13569,5.35
7,Java,8522,3.36
8,Kotlin,5258,2.07
9,Swift,4623,1.82
10,Ruby,1208,0.48


In [7]:
# Group by language and sort by date to see trend direction
if df["snapshot_date"].nunique() > 1:
    
    trend_data = []
    
    for language in df["language"].unique():
        lang_df = df[df["language"] == language].sort_values("snapshot_date")
        
        if len(lang_df) >= 2:
            first_count = lang_df.iloc[0]["total_repos"]
            last_count = lang_df.iloc[-1]["total_repos"]
            change = last_count - first_count
            change_pct = round((change / first_count) * 100, 2)
            
            trend_data.append({
                "language": language,
                "repos_first_snapshot": first_count,
                "repos_latest_snapshot": last_count,
                "change": change,
                "change_pct": change_pct,
                "trend": "rising" if change > 0 else "declining" if change < 0 else "stable"
            })
    
    df_trends = pd.DataFrame(trend_data).sort_values("change_pct", ascending=False)
    df_trends = df_trends.reset_index(drop=True)
    df_trends.index += 1
    print("Rising vs declining languages:")
    print(df_trends)

else:
    print("Only 1 day of data so far — trend analysis needs at least 2 snapshots.")
    print("Run fetch.py again tomorrow and this will show real trends!")
    df_trends = df_latest[["language"]].copy()
    df_trends["trend"] = "pending"

Only 1 day of data so far — trend analysis needs at least 2 snapshots.
Run fetch.py again tomorrow and this will show real trends!


In [8]:
# Detect languages with unusually high repo counts vs their average
if df["snapshot_date"].nunique() > 1:
    
    surge_data = []
    
    for language in df["language"].unique():
        lang_df = df[df["language"] == language].sort_values("snapshot_date")
        
        if len(lang_df) >= 2:
            avg_repos = lang_df["total_repos"].mean()
            std_repos = lang_df["total_repos"].std()
            latest_repos = lang_df.iloc[-1]["total_repos"]
            
            # Z-score: how many standard deviations above average is today?
            z_score = round((latest_repos - avg_repos) / std_repos, 2) if std_repos > 0 else 0
            
            surge_data.append({
                "language": language,
                "average_repos": round(avg_repos),
                "latest_repos": latest_repos,
                "z_score": z_score,
                "is_surge": "YES" if z_score > 1.5 else "no"
            })
    
    df_surge = pd.DataFrame(surge_data).sort_values("z_score", ascending=False)
    df_surge = df_surge.reset_index(drop=True)
    print("Surge detection (z-score > 1.5 = unusual spike):")
    print(df_surge)

else:
    print("Need more than 1 snapshot for surge detection.")
    df_surge = df_latest[["language"]].copy()
    df_surge["is_surge"] = "pending"

Need more than 1 snapshot for surge detection.


In [9]:
# Merge all analysis results together
df_analysis = df_latest[["language", "total_repos", "top_repo_name", 
                           "top_repo_stars", "top_repo_url", "snapshot_date"]].copy()

# Add share percentage
df_analysis = df_analysis.merge(
    topic_pulse[["language", "share_pct"]], on="language", how="left"
)

# Add trend info
df_analysis = df_analysis.merge(
    df_trends[["language", "trend"]], on="language", how="left"
)

# Add surge info
df_analysis = df_analysis.merge(
    df_surge[["language", "is_surge"]], on="language", how="left"
)

# Sort by total repos
df_analysis = df_analysis.sort_values("total_repos", ascending=False).reset_index(drop=True)

# Save
df_analysis.to_csv(OUTPUT_FILE, index=False)
print(f"Analysis saved to {OUTPUT_FILE}")
print(f"Rows: {len(df_analysis)}")
df_analysis

Analysis saved to ../data/analysis.csv
Rows: 10


,language,total_repos,top_repo_name,top_repo_stars,top_repo_url,snapshot_date,share_pct,trend,is_surge
0,Python,110355,NousResearch/hermes-agent,128013,https://github.com/NousResearch/hermes-agent,2026-05-01,43.52,pending,pending
1,TypeScript,49258,openclaw/openclaw,367120,https://github.com/openclaw/openclaw,2026-05-01,19.43,pending,pending
2,JavaScript,29834,affaan-m/everything-claude-code,171432,https://github.com/affaan-m/everything-claude-...,2026-05-01,11.77,pending,pending
3,Rust,15995,ultraworkers/claw-code,189525,https://github.com/ultraworkers/claw-code,2026-05-01,6.31,pending,pending
4,C++,14954,LadybirdBrowser/ladybird,62618,https://github.com/LadybirdBrowser/ladybird,2026-05-01,5.90,pending,pending
5,Go,13569,danielmiessler/Fabric,41314,https://github.com/danielmiessler/Fabric,2026-05-01,5.35,pending,pending
6,Java,8522,opendataloader-project/opendataloader-pdf,19844,https://github.com/opendataloader-project/open...,2026-05-01,3.36,pending,pending
7,Kotlin,5258,kavishdevar/librepods,26774,https://github.com/kavishdevar/librepods,2026-05-01,2.07,pending,pending
8,Swift,4623,apple/container,26263,https://github.com/apple/container,2026-05-01,1.82,pending,pending
9,Ruby,1208,antiwork/gumroad,8995,https://github.com/antiwork/gumroad,2026-05-01,0.48,pending,pending


In [10]:
print("=" * 50)
print("ANALYSIS COMPLETE")
print("=" * 50)
print(f"Date: {TODAY}")
print(f"Languages analyzed: {len(df_analysis)}")
print(f"\nTop language by repos: {df_analysis.iloc[0]['language']} ({df_analysis.iloc[0]['total_repos']:,} repos)")
print(f"Top repo by stars: {df_analysis.iloc[0]['top_repo_name']} ({df_analysis.iloc[0]['top_repo_stars']:,} stars)")
print(f"\nFile saved: {OUTPUT_FILE}")
print("=" * 50)

ANALYSIS COMPLETE
Date: 2026-05-01
Languages analyzed: 10

Top language by repos: Python (110,355 repos)
Top repo by stars: NousResearch/hermes-agent (128,013 stars)

File saved: ../data/analysis.csv
